In [1]:
!pip install -q qdrant-client sentence-transformers FlagEmbedding \
               underthesea pyvi uuid6

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.3 MB/s eta 0:00:00


# run embedding

### config

In [ ]:
from __future__ import annotations

import gc
import json
import os

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from FlagEmbedding import BGEM3FlagModel

from google.colab import userdata
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct


EMBEDDING_MODELS: dict[str, dict] = {
    "bkai": {
        "model_id":          "bkai-foundation-models/vietnamese-bi-encoder",
        "loader":            "auto",   # AutoModel + mean pool thủ công
        "needs_word_seg":    True,     # dùng field segmented_text
    },
    "dangvantuan": {
        "model_id":          "dangvantuan/vietnamese-embedding",
        "loader":            "st",     # SentenceTransformer
        "needs_word_seg":    True,     # dùng field segmented_text
    },
    "bge_m3": {
        "model_id":          "BAAI/bge-m3",
        "loader":            "flag",   # BGEM3FlagModel → encode()['dense_vecs']
        "needs_word_seg":    False,
    },
    # "gte": {
    #     "model_id":          "Alibaba-NLP/gte-multilingual-base",
    #     "loader":            "st",
    #     "needs_word_seg":    False,
    #     "trust_remote_code": True,
    # },
    "halong": {
        "model_id":          "hiieu/halong_embedding",
        "loader":            "st",
        "needs_word_seg":    False,
    },
    "aiteamvn": {
        "model_id":          "AITeamVN/Vietnamese_Embedding",
        "loader":            "st",
        "needs_word_seg":    False,
    },
}

SUPPORTED_CHUNKS: dict[str, list[int]] = {
    "bkai":        [256],
    "dangvantuan": [256],
    "halong":      [256, 512],
    "bge_m3":      [256, 512, 1024],
    # "gte":         [256, 512, 1024],
    "aiteamvn":    [256, 512, 1024],
}

FILES_MAP: dict[int, list[str]] = {
    256: [
        "full_VTT_data_qdrant_points_256.json",
        "full_VietSu_data_qdrant_points_256.json",
        "full_KhamDinh_data_qdrant_points_256.json",
        "full_DVSK_data_bkai_256.json",
    ],
    512: [
        "full_VTT_data_qdrant_points_512.json",
        "full_KhamDinh_data_qdrant_points_512.json",
        "full_VietSu_data_qdrant_points_512.json",
        "full_DVSK_data_aiteamvn_512.json",
    ],
    1024: [
        "full_VTT_data_qdrant_points_1024.json",
        "full_VietSu_data_qdrant_points_1024.json",
        "full_KhamDinh_data_qdrant_points_1024.json",
        "full_DVSK_data_aiteamvn_1024.json",
    ],
}

# Batch size theo device — CPU nhỏ hơn để tránh OOM
BATCH_SIZES: dict[str, int] = {"cuda": 32, "cpu": 8}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[DEVICE] {device.upper()}")

QDRANT_URL = userdata.get("QDRANT_URL")
API_KEY    = userdata.get("QDRANT_API_KEY")
client     = QdrantClient(url=QDRANT_URL, api_key=API_KEY, timeout=60)
print("[QDRANT] Client khởi tạo thành công.")

[DEVICE] CUDA
[QDRANT] Client khởi tạo thành công.


### Helper functions       

In [ ]:
def build_collection_name(model_key: str, chunk_size: int) -> str:
    return f"history_{model_key}_chunk_{chunk_size}"


def free_memory() -> None:
    """Giải phóng RAM và VRAM sau mỗi model."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def ensure_collection(
    client: QdrantClient,
    collection_name: str,
    vector_size: int,
) -> bool:
    """
    Tạo collection nếu chưa tồn tại.
    Trả về False nếu đã tồn tại nhưng vector size không khớp → cần skip.
    """
    if not client.collection_exists(collection_name=collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
        )
        print(f"  [QDRANT] ✓ Tạo mới: '{collection_name}'  (size={vector_size})")
        return True

    existing_size = (
        client.get_collection(collection_name).config.params.vectors.size
    )
    if existing_size != vector_size:
        print(
            f"  [QDRANT] ✗ Size mismatch '{collection_name}': "
            f"tồn tại={existing_size} ≠ model={vector_size} → BỎ QUA."
        )
        return False

    print(f"  [QDRANT] ✓ '{collection_name}' đã tồn tại, size={vector_size} hợp lệ.")
    return True


def _mean_pool(
    last_hidden_state: torch.Tensor,
    attention_mask: torch.Tensor,
) -> torch.Tensor:
    """Mean pooling có masking — bỏ padding tokens."""
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    return torch.sum(last_hidden_state * mask, dim=1) / torch.clamp(
        mask.sum(dim=1), min=1e-9
    )


def load_model(
    model_key: str,
    device: str,
) -> tuple:

    cfg    = EMBEDDING_MODELS[model_key]
    loader = cfg["loader"]
    mid    = cfg["model_id"]

    print(f"\n[LOAD] {model_key.upper()} — {mid}")

    if loader == "st":
        trust = cfg.get("trust_remote_code", False)
        model = SentenceTransformer(mid, trust_remote_code=trust, device=device)
        if hasattr(model, "get_embedding_dimension"):
            vector_size = model.get_embedding_dimension()          # ST >= 3.x
        else:
            vector_size = model.get_sentence_embedding_dimension() # ST < 3.x
        print(f"  ✓ SentenceTransformer  vector_size={vector_size}")
        # ref_tokenizer = None vì ST tự quản lý tokenizer nội bộ
        return None, model, "st", vector_size

    elif loader == "flag":
        use_fp16 = (device == "cuda")
        model    = BGEM3FlagModel(mid, use_fp16=use_fp16)
        # Dùng AutoTokenizer riêng để đếm token (metadata) — BGEM3FlagModel
        # không expose tokenizer ra ngoài một cách nhất quán
        ref_tokenizer = AutoTokenizer.from_pretrained(mid)
        # Probe vector_size bằng 1 câu dummy
        probe       = model.encode(["probe"], max_length=32)["dense_vecs"]
        vector_size = int(probe.shape[1])
        print(f"  ✓ BGEM3FlagModel  fp16={use_fp16}  vector_size={vector_size}")
        return ref_tokenizer, model, "flag", vector_size

    else:  # "auto" — chỉ có bkai
        tokenizer = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
        model     = (
            AutoModel.from_pretrained(mid, trust_remote_code=True)
            .to(device)
            .eval()
        )
        vector_size = model.config.hidden_size
        print(f"  ✓ AutoModel  vector_size={vector_size}")
        return tokenizer, model, "auto", vector_size


def embed_batch(
    texts: list[str],
    ref_tokenizer,
    model,
    model_type: str,
    device: str,
    max_length: int,
) -> tuple[list[list[float]], list[int]]:

    if model_type == "st":
        vectors = model.encode(
            texts,
            batch_size=len(texts),
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        enc = model.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_attention_mask=True,
            return_tensors="pt",
        )
        token_counts = enc["attention_mask"].sum(dim=1).int().tolist()
        return vectors.tolist(), token_counts

    elif model_type == "flag":
        result  = model.encode(texts, batch_size=len(texts), max_length=max_length)
        vectors = result["dense_vecs"]
        enc     = ref_tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_attention_mask=True,
            return_tensors="pt",
        )
        token_counts = enc["attention_mask"].sum(dim=1).int().tolist()
        return vectors.tolist(), token_counts

    else:  # "auto" — bkai
        enc = ref_tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        token_counts = enc["attention_mask"].sum(dim=1).int().tolist()
        inputs = {k: v.to(device) for k, v in enc.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        vectors = _mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
        vectors = torch.nn.functional.normalize(vectors, p=2, dim=1)
        return vectors.cpu().float().tolist(), token_counts

### collection
Xử lý danh sách chunk: Đếm token, chọn text phù hợp kiến trúc model, sinh vector và upload.   


In [4]:
def run_embedding_pipeline(
    chunked_data: list[dict],
    model_key: str,
    model_type: str,
    ref_tokenizer,
    embed_model,
    client: QdrantClient,
    collection_name: str,
    device: str,
    max_length: int,
) -> int:
    cfg        = EMBEDDING_MODELS[model_key]
    batch_size = BATCH_SIZES.get(device, 8)
    total      = len(chunked_data)

    print(f"\n{'═'*70}")
    print(f"  MODEL : {model_key.upper()}  |  TYPE : {model_type.upper()}  |  BATCH : {batch_size}")
    print(f"  COLL  : {collection_name}")
    print(f"  CHUNKS: {total:,}  |  DEVICE: {device.upper()}")
    print("═" * 70)

    points_to_upload: list[PointStruct] = []

    for batch_start in range(0, total, batch_size):
        batch_chunks = chunked_data[batch_start : batch_start + batch_size]

        texts: list[str]       = []
        valid_chunks: list[dict] = []

        for chunk in batch_chunks:
            # bkai và dangvantuan: chunk đã được word-segment sẵn
            if cfg["needs_word_seg"]:
                text = chunk.get("segmented_text") or chunk.get("embed_input", "")
            else:
                text = chunk.get("embed_input", "")

            if text:
                texts.append(text)
                valid_chunks.append(chunk)

        if not texts:
            continue

        vectors, token_counts = embed_batch(
            texts=texts,
            ref_tokenizer=ref_tokenizer,
            model=embed_model,
            model_type=model_type,
            device=device,
            max_length=max_length,
        )

        for chunk, vector, tc in zip(valid_chunks, vectors, token_counts):
            point_id = chunk.get("chunk_id") or chunk.get("id")
            points_to_upload.append(
                PointStruct(
                    id=point_id,
                    vector=vector,
                    payload={
                        "book_name":    chunk.get("book_name", ""),
                        "pages":        chunk.get("pages", []),
                        "raw_text":     chunk.get("raw_text", ""),
                        "overlap_text": chunk.get("embed_input", ""),
                        "footnotes":    chunk.get("footnotes", []),
                        "token_count":  tc,
                    },
                )
            )

        processed = min(batch_start + batch_size, total)
        if (batch_start // batch_size) % 10 == 0 or processed == total:
            print(f"    [{processed:>5}/{total}] embedded...")

    if points_to_upload:
        print(f"\n[QDRANT] Upload {len(points_to_upload):,} points → '{collection_name}'...")
        client.upload_points(
            collection_name=collection_name,
            points=points_to_upload,
            batch_size=64,
            wait=True,
        )
        print("  ✓ Upload hoàn tất!")
    else:
        print("  ⚠ Không có dữ liệu hợp lệ.")

    print("═" * 70)
    return len(points_to_upload)

### Vòng lặp duyệt qua toàn bộ model và context length tương ứng

In [5]:
variable_name = ""
for model_key in EMBEDDING_MODELS:
    allowed_chunks = SUPPORTED_CHUNKS.get(model_key, [256])

    # ── Load model ─────────────────────────────────────────────────
    try:
        ref_tokenizer, embed_model, model_type, vector_size = load_model(
            model_key, device
        )
    except Exception as e:
        print(f"\n✗ [{model_key}] Tải thất bại: {e}\n")
        continue

    # ── Chạy từng chunk_size ────────────────────────────────────────
    for chunk_size in allowed_chunks:
        files_to_process = FILES_MAP.get(chunk_size, [])
        if not files_to_process:
            continue

        collection_name = build_collection_name(model_key, chunk_size)

        # Tạo hoặc kiểm tra collection
        if not ensure_collection(client, collection_name, vector_size):
            continue   # size mismatch → skip

        # ── Chạy từng file ─────────────────────────────────────────
        for file_name in files_to_process:
            if not os.path.exists(file_name):
                print(f"  [SKIP] Không tìm thấy: {file_name}")
                continue

            print(f"\n  Đọc: {file_name}")
            with open(file_name, "r", encoding="utf-8") as f:
                raw_data = json.load(f)

            # Flatten: hỗ trợ {id, payload} lẫn flat dict
            processed_data: list[dict] = []
            for item in raw_data:
                if not isinstance(item, dict):
                    continue
                if "payload" in item:
                    flat = item["payload"].copy()
                    flat["id"] = item.get("id")
                    processed_data.append(flat)
                else:
                    processed_data.append(item)

            if not processed_data:
                print(f"  [SKIP] File rỗng hoặc sai định dạng.")
                continue

            print(f"  Loaded {len(processed_data):,} chunks → bắt đầu pipeline...")

            run_embedding_pipeline(
                chunked_data=processed_data,
                model_key=model_key,
                model_type=model_type,
                ref_tokenizer=ref_tokenizer,
                embed_model=embed_model,
                client=client,
                collection_name=collection_name,
                device=device,
                max_length=chunk_size,
            )

    # ── Giải phóng bộ nhớ ──────────────────────────────────────────
    print(f"\n  Giải phóng bộ nhớ [{model_key.upper()}]...")
    del embed_model
    if ref_tokenizer is not None:   # ST model: ref_tokenizer=None, không del
        del ref_tokenizer
    free_memory()
    print(f"  ✓ Xong.\n")

print("\n" + "═" * 70)
print("  ✅  HOÀN TẤT TOÀN BỘ QUÁ TRÌNH EMBEDDING!")
print("═" * 70)


[LOAD] BKAI — bkai-foundation-models/vietnamese-bi-encoder


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

  ✓ SentenceTransformer  vector_size=1024
  [QDRANT] ✓ 'history_aiteamvn_chunk_256' đã tồn tại, size=1024 hợp lệ.

  Đọc: full_VTT_data_qdrant_points_256.json
  Loaded 3,266 chunks → bắt đầu pipeline...

══════════════════════════════════════════════════════════════════════
  MODEL : AITEAMVN  |  TYPE : ST  |  BATCH : 32
  COLL  : history_aiteamvn_chunk_256
  CHUNKS: 3,266  |  DEVICE: CUDA
══════════════════════════════════════════════════════════════════════
    [   32/3266] embedded...
    [  352/3266] embedded...
    [  672/3266] embedded...
    [  992/3266] embedded...
    [ 1312/3266] embedded...
    [ 1632/3266] embedded...
    [ 1952/3266] embedded...
    [ 2272/3266] embedded...
    [ 2592/3266] embedded...
    [ 2912/3266] embedded...
    [ 3232/3266] embedded...
    [ 3266/3266] embedded...

[QDRANT] Upload 3,266 points → 'history_aiteamvn_chunk_256'...
  ✓ Upload hoàn tất!
══════════════════════════════════════════════════════════════════════

  Đọc: full_VietSu_data_qdrant_